Before you start running the code, it's important to select the kernel. To do this, click on Sle in the upper left corner.

In [ ]:
options(java.parameters = "-Xmx32g") 
library(rJava)
.jinit()

# Cargar librerías
library(loadeR)
library(transformeR)
library(visualizeR)

In [ ]:
df <- read.csv("data_inventory.csv")
head(df)

In [ ]:
# In this example, the ERA5-Land reanalysis dataset will be loaded.
subset.pi <- subset(df, dataset == "ERA5-Land_Iberia_day")
lon.pi <- as.character(subset.pi$endpoint)
# Doing a data Inventory allow as to see what variables the dataset has,
# the years, the latitude and longitude and the units of each variable.
str(dataInventory(lon.pi))

In [ ]:
# We select a 20-year period
years <- 1986:2005

# Adjusting lat and lon to northeast of Spain
latitude <- c(39.8, 43)
longitude <- c(-2.3, 3.6)
year.range <- paste0(min(years), "-", max(years))

In [ ]:
# Load the variables. We need maximum temperature (tasmax) and precipitation (pr) to compute the indices.   
tasmax <- loadGridData(dataset=lon.pi, var ="tasmax", years = years, latLim = latitude, lonLim = longitude)
tasmax <- gridArithmetics(tasmax, 273.15, operator="-")  # Conversion to °C

pr <- loadGridData(dataset=lon.pi, var ="pr", years = years, latLim = latitude, lonLim = longitude)
pr <- gridArithmetics(pr, 1000, operator="*")  # Conversion to mm/day


In [ ]:
# If pr < 1 mm/day, it is considered a day without precipitation from a climatological perspective, so a value of 0 is assigned.
pr$Data[pr$Data < 1] <- 0

In [ ]:
# We create a earth mask
mask <- climatology(tasmax)
mask$Data[!is.na(mask$Data)] <- 1

In [ ]:
# Function to get the months from the mobile window
moving.window <- function(month) {
  months <- c((month-1)%%12, month%%12, (month+1)%%12)  # Previous, current, and next month
  months[months == 0] <- 12  # Corrects December (when (month-1)%%12 gives 0)
  return(sort(months))
}

tmax.vm <- list()

# We convert the diary data of tasmax in a estational cicle
for(i in 1:12){
  tmax.month <- subsetGrid(tasmax, season=moving.window(i))
  tmax.month <- aggregateGrid(grid = tmax.month, aggr.y = list(FUN = "mean", na.rm = FALSE))
  tmax.clim <- climatology(tmax.month)
  tmax.vm[[i]] <- tmax.clim
}
tmax.estational.cicle <- do.call(bindGrid, c(tmax.vm, list(dimension = "time")))

In [ ]:
source("functions/fun.hottest.season.R")
hottest.season <- fun.hottest.season(tmax.estational.cicle)

In [ ]:
str(hottest.season)

In [ ]:

hs <- hottest.season
hs$Data <- hottest.season$Data[2,,]
attr(hs$Data, "dimensions") <- c("lat", "lon")

spatialPlot(hs, backdrop.theme = "coastline", 
            main=paste0("Central month of the Hottest Season"), 
            color.theme="jet.colors", set.min=0.5, set.max=12.5, at=seq(0.5,12.5,1),
            xlab="Longitude", ylab="Latitude",
            colorkey = list(space = "right",
                            title = list("Month", cex = 1.4),
                            labels = list(cex = 1.5)))

In [ ]:
# Processing data 
source("functions/fun.transform.data.R")
pr.tf <- transform.data(pr, hottest.season)
tmax.tf <- transform.data(tasmax, hottest.season)
print("Transformed data: Done")

In [ ]:
source("functions/fun.intensity.hw.hd_90_95_100.R")
intensity <- fun.intensity(pr.obs=pr.tf, tmax.obs=tmax.tf, tmax.daily=tmax.tf)

In [ ]:
source("functions/fun.categories.intensity.mean.hw.hd.R")
categories <- fun.categories(intensity)

In [ ]:
c1 <- gridArithmetics(categories$c1, mask, operator="*")
c2 <- gridArithmetics(categories$c2, mask, operator="*")
c3 <- gridArithmetics(categories$c3, mask, operator="*")
c4 <- gridArithmetics(categories$c4, mask, operator="*")

In [ ]:
spatialPlot(makeMultiGrid(c2, c4, c1, c3), 
            backdrop.theme = "coastline", 
            main=paste0("Categories of intensity of compound hot-dry and hot-wet events"), 
            as.table=TRUE, names.attr=c("Category 2", "Category 4", "Category 1", "Category 3"),
            color.theme="YlOrRd", set.min=0, set.max=4.6, at=seq(0,4.6,0.2),
            xlab="Longitude", ylab="Latitude",
            colorkey = list(space = "right",
                            title = list("Category", cex = 1.4),
                            labels = list(cex = 1.5)))

In [ ]:
source("functions/fun.duration.hw.hd.R")
duration <- d.int(intensity)

In [ ]:
saveRDS(duration, file = "save.data/duration.era5.land.rds", compress = "xz")